# Aula 2. Modelos de linguagem grandes e pequenos

O caso é um conjunto de contratos de prestação de serviços em português, o mesmo da Aula 1.
Na primeira parte, um modelo de fronteira acessado por API responde à mesma pergunta sobre um
contrato com dois prompts diferentes. Na segunda, quatro modelos de portes diferentes executam
três atividades sobre contratos com aditivos, em complexidade crescente.

Duas demonstrações, na ordem em que entram na aula. Cada uma imprime um número medido pela
própria célula.

- Demonstração 1, execução de tarefa por instrução: a chamada por API e o efeito do prompt sobre a resposta.
- Demonstração 2, três atividades em quatro modelos: extração, pergunta com cálculo e detecção de contradição, e em qual delas cada modelo para de acertar.

A segunda demonstração usa a GPU deste Colab. Confira no menu Ambiente de execução que o tipo de hardware é GPU antes de rodar a primeira célula, porque trocar o ambiente no meio reinicia o kernel.

---

Material desenvolvido por
**prof. Thiago Nascimento Nogueira**
e-mail: profthiago.nogueira@fiap.com.br
linkedIn: https://www.linkedin.com/in/thnogueira/

A célula abaixo instala a biblioteca da API e pede a chave, que é fornecida em aula e não fica gravada no notebook.

In [ ]:
!pip install -q anthropic

import getpass
import anthropic

client = anthropic.Anthropic(api_key=getpass.getpass("Cole a chave fornecida em aula: "))

---
## 1. Demonstração 1, execução de tarefa por instrução

Um modelo pré-treinado executa a tarefa descrita no prompt, sem treino e sem conjunto rotulado. Esta demonstração mostra a chamada por API e mede o efeito do prompt sobre a resposta, no conteúdo e no formato.

### 1.1 Anatomia da chamada

A chamada leva o nome do modelo, a instrução de sistema, a lista de mensagens e o teto de tokens de saída. A resposta traz o texto, o motivo pelo qual a geração parou e a contagem de tokens de entrada e de saída, que é a unidade em que o uso é cobrado.

In [ ]:
MODELO = "claude-haiku-4-5"

resposta = client.messages.create(
    model=MODELO,
    max_tokens=300,
    thinking={"type": "disabled"},
    system="Você responde em português, de forma direta.",
    messages=[{"role": "user", "content": "Em duas frases, o que é um contrato de prestação de serviços?"}])

print("texto devolvido:")
print(resposta.content[0].text)
print()
print("motivo da parada:", resposta.stop_reason)
print("tokens de entrada:", resposta.usage.input_tokens)
print("tokens de saída:", resposta.usage.output_tokens)

### 1.2 Efeito do prompt sobre a resposta

Afirmação: o prompt decide o que o modelo confere antes de responder e a forma em que a resposta chega.

A pergunta é uma só, sobre um contrato de 14 cláusulas: qual é o valor global do contrato. A cláusula 14 declara um valor global, e as cláusulas de vigência e de remuneração dão outro. Dois prompts levam a mesma pergunta e o mesmo documento.

- Prompt A: os documentos e a pergunta.
- Prompt B: uma regra de verificação (conferir se as cláusulas concordam entre si e apontar divergências) e um formato de saída, um objeto JSON com os campos `valor` e `justificativa`.

A célula imprime a resposta de cada prompt e indica, para cada uma, se aponta a divergência e se chega no formato pedido.

In [ ]:
import json
import requests

BASE = "https://raw.githubusercontent.com/thiagonogueira/datasets/main/mba-ai-engineering/generative-models/aula2/"
ESCADA = requests.get(BASE + "aula2-escada.json").json()
CASO = [caso for caso in ESCADA if caso["id"] == "C2"][0]

print(CASO["documentos"])

Os dois prompts. A pergunta e o documento são os mesmos nos dois.

In [ ]:
PERGUNTA = "Qual é o valor global deste contrato?"

PROMPT_A = "Documentos:\n{documentos}\n\nPergunta: {pergunta}"

PROMPT_B = (
    "Você responde a perguntas sobre contratos usando apenas os documentos fornecidos. "
    "Antes de responder, verifique se as cláusulas concordam entre si. "
    "Se houver divergência entre partes do documento, aponte-a com os valores envolvidos.\n"
    "Responda com um único objeto JSON, sem texto antes ou depois, com estes campos:\n"
    '  {"valor": "valor global, ou os valores em divergência", "justificativa": "cláusulas usadas e a conferência feita"}\n\n'
    "Documentos:\n{documentos}\n\nPergunta: {pergunta}")

def ler_json(texto):
    posicao_abre = texto.find("{")
    posicao_fecha = texto.rfind("}")
    if posicao_abre < 0 or posicao_fecha < posicao_abre:
        return None
    try:
        return json.loads(texto[posicao_abre:posicao_fecha + 1])
    except ValueError:
        return None


resultado = {}
for nome, prompt in (("A", PROMPT_A), ("B", PROMPT_B)):
    resposta = client.messages.create(
        model=MODELO, max_tokens=1500, thinking={"type": "disabled"},
        messages=[{"role": "user", "content": prompt.replace("{documentos}", CASO["documentos"]).replace("{pergunta}", PERGUNTA)}])
    texto = resposta.content[0].text.strip()
    apontou = "48.000" in texto
    objeto = ler_json(texto)
    formato = objeto is not None and "valor" in objeto and "justificativa" in objeto
    resultado[nome] = (apontou, formato)
    print("PROMPT", nome)
    print(texto)
    print()
    print(f"aponta a divergência: {'sim' if apontou else 'não'}   no formato pedido: {'sim' if formato else 'não'}")
    print()

In [ ]:
assert resultado["A"] == (False, False)
assert resultado["B"] == (True, True)

**Leitura:** as duas respostas. Com o prompt A o modelo responde pela cláusula que contém a resposta, e o valor errado escrito no contrato passa. Com o prompt B o modelo compara as cláusulas antes de responder e a resposta chega num objeto que um programa lê sem tratamento. O modelo foi o mesmo nas duas chamadas. O que mudou foi o texto que antecede a pergunta.

Nesta demonstração o modelo ficou fixo e o prompt mudou. A próxima mantém o prompt fixo e troca o modelo.

---
## 2. Demonstração 2, três atividades em quatro modelos

Três atividades sobre contratos de prestação de serviços, em complexidade crescente. Em cada uma, os quatro modelos recebem a mesma instrução e os mesmos documentos, e a medida é o acerto contra uma referência conferida à mão. Dois modelos são abertos, de 1,7 e 4 bilhões de parâmetros, executados na GPU deste Colab. Dois são de fronteira, acessados por API, de dois portes de preço.

### 2.1 Os modelos

A célula abaixo carrega os dois modelos abertos na GPU. O modo de raciocínio interno do Qwen3 fica desligado, para que a resposta seja comparável à dos modelos por API, que também respondem sem raciocínio estendido. A geração é gulosa, o que torna a resposta dos modelos abertos a mesma em toda execução.

In [ ]:
!pip install -q transformers accelerate

import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(torch.cuda.get_device_name(0))

tokenizador_17 = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
modelo_17 = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B", dtype=torch.bfloat16, device_map="cuda")
tokenizador_4 = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
modelo_4 = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", dtype=torch.bfloat16, device_map="cuda")

Daqui em diante o mesmo procedimento roda para quatro modelos em três atividades, então ele vira duas funções: uma gera a resposta num modelo aberto, a outra num modelo por API. As duas devolvem o texto e o tempo. A célula seguinte lista os quatro modelos na ordem em que aparecem nas tabelas.

In [ ]:
def gerar_local(tokenizador, modelo, prompt):
    mensagens = [{"role": "user", "content": prompt}]
    texto_entrada = tokenizador.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    entrada = tokenizador(texto_entrada, return_tensors="pt").to("cuda")
    inicio = time.time()
    with torch.no_grad():
        gerado = modelo.generate(**entrada, max_new_tokens=700, do_sample=False)
    segundos = time.time() - inicio
    tokens_novos = gerado[0][entrada["input_ids"].shape[1]:]
    return tokenizador.decode(tokens_novos, skip_special_tokens=True), segundos


def gerar_api(nome_modelo, prompt):
    inicio = time.time()
    retorno = client.messages.create(
        model=nome_modelo, max_tokens=3000, thinking={"type": "disabled"},
        messages=[{"role": "user", "content": prompt}])
    segundos = time.time() - inicio
    return retorno.content[0].text, segundos


def gerar(chave, prompt):
    if chave == "Qwen3-1.7B":
        return gerar_local(tokenizador_17, modelo_17, prompt)
    if chave == "Qwen3-4B":
        return gerar_local(tokenizador_4, modelo_4, prompt)
    if chave == "Haiku 4.5":
        return gerar_api("claude-haiku-4-5", prompt)
    return gerar_api("claude-sonnet-5", prompt)


MODELOS = ["Qwen3-1.7B", "Qwen3-4B", "Haiku 4.5", "Sonnet 5"]
resumo = {}

### 2.2 Atividade 1, extração de campos

O que a atividade pede:

- Ler um contrato curto e devolver cinco campos em JSON: nome, CPF, data de início, valor mensal e prazo em meses.
- Cada valor no formato declarado na instrução (CPF só com dígitos, data em AAAA-MM-DD, valor como número).
- Campo que o contrato não informa recebe null.

O que ela exige do modelo:

- Localizar cada dado no texto e transcrevê-lo no formato pedido.
- Nenhum cálculo e nenhuma comparação entre partes do documento.

A medida é o número de campos iguais aos da referência, em quatro contratos: dois na forma canônica, com rótulos explícitos, e dois na forma corrida, com os dados dentro do texto.

In [ ]:
import json
import requests

BASE_AULA1 = "https://raw.githubusercontent.com/thiagonogueira/datasets/main/mba-ai-engineering/generative-models/aula1/"
BASE = "https://raw.githubusercontent.com/thiagonogueira/datasets/main/mba-ai-engineering/generative-models/aula2/"

TODOS_CONTRATOS = requests.get(BASE_AULA1 + "aula1-contratos.json").json()

# dois contratos de cada forma
CONTRATOS = []
canonica = 0
corrida = 0
for contrato in TODOS_CONTRATOS:
    if contrato["conjunto"] != "principal":
        continue
    if contrato["forma"] == "canônica" and canonica < 2:
        CONTRATOS.append(contrato)
        canonica += 1
    if contrato["forma"] == "corrida" and corrida < 2:
        CONTRATOS.append(contrato)
        corrida += 1

print("exemplo, forma corrida:")
print(CONTRATOS[2]["texto"])
print()
print("referência:", json.dumps(CONTRATOS[2]["referencia"], ensure_ascii=False))

In [ ]:
INSTRUCAO_EXTRACAO = (
    "Você extrai campos de contratos de prestação de serviços e devolve um único objeto JSON, sem texto antes ou depois.\n"
    "Campos e formatos:\n"
    "- nome: nome completo do CONTRATANTE, como está no contrato\n"
    "- cpf: somente os 11 dígitos, sem pontos nem traço\n"
    "- inicio: data de início da vigência no formato AAAA-MM-DD\n"
    "- valor_mensal: número, sem símbolo de moeda (3200.0 para R$ 3.200,00)\n"
    "- prazo_meses: número inteiro de meses\n"
    "Use somente informações presentes no contrato. Se um campo não estiver no contrato, use null.\n\n"
    "Contrato:\n{texto}")


def ler_json(texto):
    posicao_abre = texto.find("{")
    posicao_fecha = texto.rfind("}")
    if posicao_abre < 0 or posicao_fecha < posicao_abre:
        return None
    try:
        return json.loads(texto[posicao_abre:posicao_fecha + 1])
    except ValueError:
        return None


for chave in MODELOS:
    print(chave)
    campos_certos = 0
    segundos_total = 0.0
    for contrato in CONTRATOS:
        texto, segundos = gerar(chave, INSTRUCAO_EXTRACAO.replace("{texto}", contrato["texto"]))
        segundos_total += segundos
        objeto = ler_json(texto)
        certos_no_contrato = 0
        for campo in contrato["referencia"]:
            if objeto is not None and str(objeto.get(campo)) == str(contrato["referencia"][campo]):
                certos_no_contrato += 1
        campos_certos += certos_no_contrato
        print(f"  {contrato['id']} ({contrato['forma']:8}) {certos_no_contrato} de 5 campos  {segundos:5.1f} s  {texto.replace(chr(10), ' ')}")
    resumo[("extração", chave)] = (campos_certos, len(CONTRATOS) * 5, segundos_total)
    print(f"  total: {campos_certos} de {len(CONTRATOS) * 5} campos")
    print()

**Análise:** os quatro totais, e o campo errado quando houver. A tarefa é localizar e transcrever no formato declarado.

### 2.3 Atividade 2, pergunta com cálculo sobre o contrato

O que a atividade pede:

- Responder a uma pergunta de gestor sobre um contrato de 14 cláusulas, com aditivos em dois dos casos.
- A resposta é um valor, e a instrução pede o raciocínio antes dela, em JSON com os campos `raciocinio` e `resposta`.

O que ela acrescenta em relação à extração:

- Juntar dados de cláusulas diferentes e calcular (caso 1: 18 meses vezes R$ 3.200,00).
- Aplicar a precedência de um aditivo sobre o contrato e dividir a vigência em trechos (caso 3: o valor muda no sétimo mês).
- Três documentos, três trechos de valor e um reajuste percentual (caso 5).

A medida é a resposta final conter o valor da referência. Os casos vêm do arquivo `aula2-escada.json`: cada um tem os documentos, a pergunta, a referência e as `chaves`, os trechos que a resposta precisa conter.

In [ ]:
ESCADA = requests.get(BASE + "aula2-escada.json").json()
POR_ID = {}
for caso in ESCADA:
    POR_ID[caso["id"]] = caso

INSTRUCAO_CONTRATO = (
    "Você responde a perguntas sobre contratos de prestação de serviços e seus termos aditivos, usando apenas os documentos fornecidos. Regras:\n"
    "- Localize as cláusulas e os itens de aditivo que tratam do assunto e use somente o que eles afirmam.\n"
    "- Um aditivo prevalece sobre o contrato original e sobre aditivos anteriores naquilo que alterar expressamente. O que não foi alterado continua em vigor.\n"
    "- Quando a resposta exigir cálculo ou contagem de prazo, faça o cálculo passo a passo, trecho por trecho, antes de responder.\n"
    "- Se houver divergência entre partes dos documentos, aponte-a com os valores ou datas envolvidos.\n"
    "- Responda com um único objeto JSON, sem texto antes ou depois, com estes campos:\n"
    '  {"raciocinio": "passos e cláusulas usadas", "resposta": "resposta final, curta e objetiva"}\n\n'
    "Documentos:\n{documentos}\n\nPergunta: {pergunta}")


def conferir_resposta(caso, texto):
    objeto = ler_json(texto)
    resposta_final = texto
    if objeto is not None and "resposta" in objeto:
        resposta_final = str(objeto["resposta"])
    acertou = True
    for grupo in caso["chaves"]:
        achou_no_grupo = False
        for trecho in grupo:
            if trecho.lower() in resposta_final.lower():
                achou_no_grupo = True
        if not achou_no_grupo:
            acertou = False
    for trecho in caso["nega"]:
        if trecho.lower() in resposta_final.lower():
            acertou = False
    return acertou, resposta_final


caso = POR_ID["C3"]
print("exemplo, caso 3. Pergunta:", caso["pergunta"])
print("referência:", caso["referencia"])

In [ ]:
CASOS_CALCULO = ["C1", "C3", "C5"]

for chave in MODELOS:
    print(chave)
    acertos = 0
    segundos_total = 0.0
    for id_caso in CASOS_CALCULO:
        caso = POR_ID[id_caso]
        texto, segundos = gerar(chave, INSTRUCAO_CONTRATO.replace("{documentos}", caso["documentos"]).replace("{pergunta}", caso["pergunta"]))
        segundos_total += segundos
        acertou, resposta_final = conferir_resposta(caso, texto)
        if acertou:
            acertos += 1
        print(f"  {id_caso}  {'certo ' if acertou else 'errado'}  {segundos:5.1f} s  {resposta_final.replace(chr(10), ' ')}")
    resumo[("cálculo", chave)] = (acertos, len(CASOS_CALCULO), segundos_total)
    print(f"  total: {acertos} de {len(CASOS_CALCULO)}")
    print()

**Análise:** o caso em que cada modelo para de acertar, e a resposta errada dele: se somou os trechos da vigência ou tomou um valor mensal como total. Compare o tempo por caso dos modelos abertos com o dos modelos por API.

### 2.4 Atividade 3, detecção de contradição

O que a atividade pede:

- Verificar se as partes de um contrato concordam entre si e apontar a divergência com os valores envolvidos.
- Nos casos com aditivos, dizer se alguma cláusula não alterada ficou em conflito com o que o aditivo estabelece.

O que ela acrescenta em relação ao cálculo:

- Comparar partes distantes do documento: a cláusula 14 declara um valor global que contradiz as cláusulas 2 e 3 (caso 2).
- Inferir um conflito que nenhum documento declara: o valor fixado por aditivo é incompatível com a cláusula de reajuste, que ninguém revogou (caso 6).
- Uma cadeia de três reajustes compostos, com troca de índice e acréscimo fixo, em 48 mensalidades (caso 8). Nos modelos por API o caso 8 roda cinco vezes, porque a resposta deles varia entre chamadas.

A medida é a mesma da atividade 2.

In [ ]:
CASOS_CONTRADICAO = ["C2", "C6", "C8"]
REPETICOES_C8 = 5

for chave in MODELOS:
    print(chave)
    acertos = 0.0
    segundos_total = 0.0
    for id_caso in CASOS_CONTRADICAO:
        caso = POR_ID[id_caso]
        prompt = INSTRUCAO_CONTRATO.replace("{documentos}", caso["documentos"]).replace("{pergunta}", caso["pergunta"])
        if id_caso == "C8" and chave in ("Haiku 4.5", "Sonnet 5"):
            acertos_c8 = 0
            for repeticao in range(REPETICOES_C8):
                texto, segundos = gerar(chave, prompt)
                segundos_total += segundos / REPETICOES_C8
                acertou, resposta_final = conferir_resposta(caso, texto)
                if acertou:
                    acertos_c8 += 1
                print(f"  {id_caso} execução {repeticao + 1}  {'certo ' if acertou else 'errado'}  {segundos:5.1f} s  {resposta_final.replace(chr(10), ' ')}")
            acertos += acertos_c8 / REPETICOES_C8
            print(f"  {id_caso}  {acertos_c8} de {REPETICOES_C8} execuções")
        else:
            texto, segundos = gerar(chave, prompt)
            segundos_total += segundos
            acertou, resposta_final = conferir_resposta(caso, texto)
            if acertou:
                acertos += 1
            print(f"  {id_caso}  {'certo ' if acertou else 'errado'}  {segundos:5.1f} s  {resposta_final.replace(chr(10), ' ')}")
    resumo[("contradição", chave)] = (acertos, len(CASOS_CONTRADICAO), segundos_total)
    print(f"  total: {acertos:g} de {len(CASOS_CONTRADICAO)}")
    print()

**Análise:** no caso 2, se o modelo aceitou o valor escrito na cláusula 14 ou o conferiu contra as cláusulas 2 e 3. No caso 6, se encontrou o conflito com a cláusula de reajuste. No caso 8, a variação entre as cinco execuções de cada modelo de API.

### 2.5 Resumo das três atividades

A tabela reúne o acerto e o tempo médio por caso de cada modelo em cada atividade.

In [ ]:
print(f"{'atividade':14}", end="")
for chave in MODELOS:
    print(f"{chave:>22}", end="")
print()
for atividade in ("extração", "cálculo", "contradição"):
    print(f"{atividade:14}", end="")
    for chave in MODELOS:
        acertos, total, segundos = resumo[(atividade, chave)]
        if acertos == int(acertos):
            texto_acertos = f"{int(acertos):>10}"
        else:
            texto_acertos = f"{acertos:>10.1f}"
        print(f"{texto_acertos} de {total:<2} {segundos / total:5.1f} s", end="")
    print()

In [ ]:
assert resumo[("extração", "Sonnet 5")][0] >= 18
assert resumo[("cálculo", "Qwen3-1.7B")][0] <= 1
assert resumo[("cálculo", "Sonnet 5")][0] == 3
assert resumo[("contradição", "Qwen3-1.7B")][0] <= 1
assert resumo[("contradição", "Haiku 4.5")][0] <= 2.6
assert resumo[("contradição", "Sonnet 5")][0] >= 2.6

**Leitura:**

- Na extração, a distância entre os quatro modelos. A tarefa é localizar e transcrever.
- No cálculo, o caso em que cada modelo para. O que muda de um caso para o outro é a operação exigida: dividir a vigência em trechos, aplicar um reajuste.
- Na contradição, quais modelos comparam partes distantes do documento, e o que a repetição do caso 8 mostra sobre os dois modelos de API.

## Resumo

- Na Demonstração 1 o modelo ficou fixo e o prompt mudou: a regra de verificação decidiu o conteúdo da resposta, e o formato declarado decidiu a forma.
- Na Demonstração 2 o prompt ficou fixo e o modelo mudou: cada atividade acrescenta uma operação, e cada modelo para numa delas. A atividade em que a tarefa está é o que decide o porte, e a repetição decide entre dois modelos que acertam uma vez.